# Рынок новых автомобилей Казахстана в 2019 году: очистка данных и позиция «Меркур Авто»

**Бизнес-контекст.** Автодилерский холдинг анализирует продажи новых
автомобилей в Казахстане за девять месяцев 2019 года. Один из автоцентров
холдинга — «Меркур Авто» (Mercur Auto) — хочет понять своё положение на рынке
и найти точки роста.

**Проблема данных.** Выгрузка пришла в «операционном» виде: десятичная
запятая, 80+ вариантов записи коробки передач, одни и те же привод, страна и
компания записаны по-разному, в объёме двигателя встречаются мощность и
ошибки автозаполнения, часть значений сдвинута на соседний столбец. Без
очистки любые доли и средние будут искажены.

**Цели.**

1. *Техническая* — привести датасет к виду, пригодному для анализа и загрузки
   в BI, и автоматизировать очистку скриптом для похожих выгрузок
   (`src/kz_auto_market/cleaning.py`, команда `kz-auto-clean`).
2. *Аналитическая* — провести разведочный анализ всего рынка и оценить долю,
   структуру продаж и ключевые показатели «Меркур Авто» в сравнении с рынком,
   определить сильные и слабые стороны и предложить решения.

Каждый шаг очистки ниже выполняется функцией из `cleaning.py` — тем же кодом,
что и в скрипте; в конце раздела 2 результат сверяется с сохранённым
`data/processed/autokz2019_clean.csv`.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from kz_auto_market import cleaning

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_PATH = ROOT / cleaning.RAW_PATH
CLEAN_PATH = ROOT / cleaning.CLEAN_PATH
MERCUR = cleaning.FOCUS_COMPANY
PREMIUM_BRANDS = [
    'Audi',
    'BMW',
    'Cadillac',
    'Infiniti',
    'Jaguar',
    'Land Rover',
    'Lexus',
    'Mercedes-Benz',
    'Mini',
    'Porsche',
    'Volvo',
]
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)
plt.rcParams['figure.dpi'] = 90

## 1. Загрузка и первичный осмотр данных

In [2]:
# Разделитель — точка с запятой, десятичный знак — запятая: без decimal=','
# цены и количество прочитались бы строками
raw = cleaning.read_raw(RAW_PATH)
print(f'Строк: {raw.shape[0]}, столбцов: {raw.shape[1]}')
raw.dtypes.value_counts()

Строк: 39966, столбцов: 25


str        21
float64     3
int64       1
Name: count, dtype: int64

In [3]:
raw.head(3).T

,0,1,2
Год,2019,2019,2019
Месяц,Май,Август,Апрель
Компания,Mercur Auto,Mercur Auto,Mercur Auto
Бренд,Audi,Audi,Audi
Модель,A3,A3,A4
Модификация,TFSI,TFSI,TFSI
Год выпуска,2018,2018,2018
Страна-производитель,Германия,Германия,Германия
Вид топлива,Бензин,Бензин,Бензин
"Объём двиг, л,","1,4","1,4","1,4"


In [4]:
missing = pd.DataFrame(
    {
        'пропусков': raw.isna().sum(),
        'доля, %': (raw.isna().mean() * 100).round(1),
        'уникальных': raw.nunique(),
    }
)
missing.sort_values('пропусков', ascending=False)

,пропусков,"доля, %",уникальных
Форма расчета,25928,64.9,21
Тип клиента,7047,17.6,9
Сегмент,6761,16.9,119
Тип привода,4289,10.7,26
"Объём двиг, л,",4258,10.7,114
Модификация,3591,9.0,1383
Коробка передач,3255,8.1,83
Вид топлива,3140,7.9,10
Год выпуска,501,1.3,9
Количество,6,0.0,56


Столбцы из плана работы удаляются: они дублируют другие признаки, не нужны
для анализа рынка или содержат критичное число пропусков. В выгрузке у
некоторых из них другие названия:

| В плане | В выгрузке | Причина |
|---------|------------|---------|
| Дилер | Наименование дилерского центра | повторяет «Компания» + «Регион» |
| Тип платежа | Форма расчета | 65% пропусков, 21 вариант записи |
| Сегмент | Сегмент | 17% пропусков и 119 вариантов; сегментацию дают «Сегментация 2013» и «Класс 2013» |
| Сегмент ЕПР | Сегментация Eng | буквенные коды тех же классов, что «Класс 2013» |
| Локализация | Локализация производства | однозначно следует из страны производства |
| Тип клиента | Тип клиента | 18% пропусков |
| Модификация | Модификация | свободный текст комплектации |

Названия «Стоимость», «Итоговая стоимость» и «Сегмент 2013» из плана
соответствуют столбцам «Цена, USD», «Продажа, USD» и «Сегментация 2013».

In [5]:
data = cleaning.rename_columns(cleaning.drop_unused_columns(raw))
print(list(data.columns))

['year', 'month', 'company', 'brand', 'model', 'release_year', 'country', 'fuel', 'engine_volume', 'transmission', 'drive', 'region', 'quantity', 'price_usd', 'sale_usd', 'province', 'segment_2013', 'class_2013']


In [6]:
def volume(df):
    """Строки, проданные автомобили и выручка таблицы."""
    return pd.Series(
        {
            'строк': len(df),
            'автомобилей': df['quantity'].sum(),
            'выручка, млн USD': round(df['sale_usd'].sum() / 1e6, 1),
        }
    )


print(f'Полностью пустых строк: {data.isna().all(axis=1).sum()}')
print(f'Явных дубликатов: {data.duplicated().sum()}')
before_dedup = data
data = cleaning.drop_empty_and_duplicates(data)
pd.DataFrame({'до': volume(before_dedup), 'после': volume(data)})

Полностью пустых строк: 0


Явных дубликатов: 23471


,до,после
строк,39966.0,16495.0
автомобилей,53874.0,30335.0
"выручка, млн USD",1349.2,677.6


Пустых строк нет, явных дубликатов — 23 471, больше половины строк. Они
удалены по плану работы. Важное ограничение: в выгрузке нет VIN или номера
сделки, поэтому две одинаковые строки могут быть двумя проданными
автомобилями одной модели в одном месяце. Удаление дубликатов уменьшает
абсолютные объёмы рынка (с 53.9 до 30.3 тыс. автомобилей), поэтому ниже
выводы строятся на долях и средних, а в разделе 4 проверено, что доля
«Меркур Авто» от удаления дубликатов почти не меняется.

**Решение для источника данных:** добавить в выгрузку идентификатор сделки
или VIN — тогда настоящие дубли можно будет отличить от повторных продаж.